<a href="https://colab.research.google.com/github/varunkshatriya/flyrank-ml-internship-starter-clone/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## 1. Two paper findings + my methodology questions

### Finding 1 — The label comes from observed trend information

The FlyRank workflow defines the declining-page label from `trend_direction == "down"`.

This is important because the target is based on an observed outcome rather than a model-generated prediction. The label should therefore remain separate from the feature set during training.

**Methodology question:** How stable is this label across different time windows, and would the same validation result hold if the observation window changed?

### Finding 2 — Client-level validation is used to reduce leakage

The reference workflow holds out clients rather than randomly splitting individual pages. This prevents pages belonging to the same client from appearing in both training and test sets.

This makes the evaluation more realistic because pages from one client can share characteristics.

**Methodology question:** Would the model maintain its performance under a time-aware split where future observations are held out rather than simply unseen clients?

### Constructive interpretation

The validation design provides a useful test of whether the model generalizes beyond the clients used for training. However, a client-level split does not by itself establish performance under future time periods, so a time-aware validation would be a useful additional check.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [1]:
# ============================================================
# ML-09 SECTION 2 — HONEST VALIDATION
# ============================================================

from pathlib import Path
import subprocess
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# ------------------------------------------------------------
# 1. Locate dataset
# ------------------------------------------------------------

matches = list(
    Path("/content").rglob(
        "content_refresh_anonymized.csv"
    )
)

if not matches:

    repo_path = Path("/content/flyrank-starter")

    if not repo_path.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "https://github.com/flyrank-bih/flyrank-ml-internship-starter.git",
                str(repo_path)
            ],
            check=True
        )

    matches = list(
        repo_path.rglob(
            "content_refresh_anonymized.csv"
        )
    )

if not matches:
    raise FileNotFoundError(
        "Could not find content_refresh_anonymized.csv"
    )

data_path = matches[0]

print("Dataset:", data_path)

df = pd.read_csv(data_path)

print("Shape:", df.shape)

# ------------------------------------------------------------
# 2. Target
# ------------------------------------------------------------

if "trend_direction" not in df.columns:
    raise ValueError(
        "trend_direction not found."
    )

y = (
    df["trend_direction"]
    .astype(str)
    .str.lower()
    .eq("down")
    .astype(int)
)

# ------------------------------------------------------------
# 3. Observed features only
# ------------------------------------------------------------

candidate_features = [
    "ctr",
    "avg_position",
    "search_volume",
    "impressions_90d",
    "clicks_90d",
    "content_age_days"
]

features = [
    c for c in candidate_features
    if c in df.columns
]

print("Features:", features)

# ------------------------------------------------------------
# 4. Client grouping
# ------------------------------------------------------------

group_candidates = [
    "client_id",
    "client_key",
    "client",
    "client_hash",
    "site_id",
    "domain_id"
]

group_col = next(
    (
        c for c in group_candidates
        if c in df.columns
    ),
    None
)

if group_col is None:
    print(df.columns.tolist())
    raise ValueError(
        "Could not identify the client grouping column."
    )

print("Grouping column:", group_col)

X = df[features].copy()
groups = df[group_col]

# ------------------------------------------------------------
# 5. Client-level split
# ------------------------------------------------------------

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("\nCLIENT-LEVEL SPLIT")
print("------------------")
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print(
    "Training clients:",
    groups.iloc[train_idx].nunique()
)
print(
    "Test clients:",
    groups.iloc[test_idx].nunique()
)

overlap = set(
    groups.iloc[train_idx]
) & set(
    groups.iloc[test_idx]
)

print("Client overlap:", len(overlap))

assert len(overlap) == 0

# ------------------------------------------------------------
# 6. Train
# ------------------------------------------------------------

model = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    )
])

model.fit(
    X_train,
    y_train
)

scores = model.predict_proba(
    X_test
)[:, 1]

# ------------------------------------------------------------
# 7. Precision@50
# ------------------------------------------------------------

def precision_at_k(
    y_true,
    scores,
    k=50
):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    k = min(k, len(y_true))

    order = np.argsort(
        -scores
    )

    return float(
        y_true[order[:k]].mean()
    )


honest_p50 = precision_at_k(
    y_test,
    scores,
    50
)

print(
    "\nHonest client-holdout Precision@50:",
    round(honest_p50, 4)
)

Dataset: /content/flyrank-starter/data/raw/content_refresh_anonymized.csv
Shape: (30000, 44)
Features: ['ctr', 'avg_position', 'search_volume', 'impressions_90d', 'clicks_90d', 'content_age_days']
Grouping column: client_id

CLIENT-LEVEL SPLIT
------------------
Training rows: 23837
Test rows: 6163
Training clients: 25
Test clients: 7
Client overlap: 0

Honest client-holdout Precision@50: 0.68


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [2]:
# ============================================================
# ML-09 SECTION 3 — LEAKAGE AUDIT
# ============================================================

print("## Leakage audit")

# ------------------------------------------------------------
# Features actually used by the final model
# ------------------------------------------------------------

print("Final feature set:")
for feature in features:
    print(" -", feature)

# ------------------------------------------------------------
# Known leakage / label-derived columns
# ------------------------------------------------------------

forbidden_columns = {
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "health_score",
    "needs_ctr_fix",
    "is_quick_win",
    "baseline_score",
    "model_score",
    "prediction",
    "predicted",
    "label"
}

leaked_features = (
    set(features)
    & forbidden_columns
)

print("\nForbidden columns detected in feature set:")
print(leaked_features)

assert not leaked_features, (
    "Potential label/product-flag leakage detected."
)

# ------------------------------------------------------------
# Check target is not inside X
# ------------------------------------------------------------

assert "trend_direction" not in X.columns

# ------------------------------------------------------------
# Check train/test client separation
# ------------------------------------------------------------

train_clients = set(
    groups.iloc[train_idx]
)

test_clients = set(
    groups.iloc[test_idx]
)

client_overlap = (
    train_clients
    & test_clients
)

assert len(client_overlap) == 0

print("\nLeakage checks")
print("--------------")
print("Label-derived feature leakage: PASS")
print("Client overlap: PASS")
print("Future target used as feature: PASS")

print(
    "\nConclusion: No obvious label-derived or "
    "client-overlap leakage was detected in the "
    "final feature set."
)

## Leakage audit
Final feature set:
 - ctr
 - avg_position
 - search_volume
 - impressions_90d
 - clicks_90d
 - content_age_days

Forbidden columns detected in feature set:
set()

Leakage checks
--------------
Label-derived feature leakage: PASS
Client overlap: PASS
Future target used as feature: PASS

Conclusion: No obvious label-derived or client-overlap leakage was detected in the final feature set.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## 4. Claim rewrite

### Original bold claim

> The model predicts which pages will decline in Google Search.

### Safer version

> On the evaluated held-out data, the model measured a Precision@50 of the reported value and provides directional decision-support for prioritizing pages for review.

### Why I changed the wording

The safer statement describes what was actually measured on the evaluation data.

It does not claim that the model understands or predicts Google's ranking algorithm.

The result should be interpreted as observed and measured performance on this dataset and validation design, rather than as a guarantee of future performance.


1. Paper findings                 ✅
2. Honest client split            ✅
3. Leakage audit                  ✅
4. Claim rewrite                  ✅
5. Self-check                     ✅